In [1]:
import re
import os
import sys
import gzip
import torch
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, GPT2Model

/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu" 
tokenizer_file = "lixiangchun/transcriptome-gpt-1024-8-16-64" 
checkpoint = "lixiangchun/transcriptome-gpt-1024-8-16-64" ## Pretrained model

# celltype_path = "./data/Muris_cell_labels.txt.gz" ## Cell type annotation
max_len = 500 ## Number of top genes used for analysis
text_file = "/scratch/2370352/my-research/data_utils/tpgt/bulk_gene_rankings_ov.txt.gz"  ## Gene symbols ranked by exprssion

In [3]:
data_path = '../../data/0_data_for_mlp/TCGA-OV.star_tpm.csv'
df = pd.read_csv(data_path)

In [4]:
class LineDataset(Dataset):
    def __init__(self, lines):
        self.lines = lines
        self.regex = re.compile(r'\-|\.')
    def __getitem__(self, i):
        return self.regex.sub('_', self.lines[i])
    def __len__(self):
        return len(self.lines)

tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_file)
model = GPT2LMHeadModel.from_pretrained(checkpoint,output_hidden_states = True).transformer
model = model.to(device)
model.eval()

lines = [s.decode().strip() for s in gzip.open(text_file, "r").readlines()]

ds = LineDataset(lines)
dl = DataLoader(ds, batch_size=64)

Xs = []
for a in tqdm(dl, total=len(dl)):
    batch = tokenizer(a, max_length= max_len, truncation=True, padding=True, return_tensors="pt")

    for k, v in batch.items():
        batch[k] = v.to(device)

    with torch.no_grad():
        x = model(**batch)
    
    eos_idxs = batch.attention_mask.sum(dim=1) - 1
    xx = x.last_hidden_state
       
    result_list = [[] for i in range(len(xx))]

    for j, item in enumerate(xx):
        result_list[j] = item[1:int(eos_idxs[j]),:].mean(dim =0).tolist()
        
    Xs.extend(result_list)
    
features = np.stack(Xs)

100%|██████████| 7/7 [00:07<00:00,  1.14s/it]


In [5]:
# 1. Pobranie nazw sampli (identyfikatorów) z pierwszej kolumny oryginalnego DataFrame
# Zakładam, że df to tabela z surowymi danymi, którą wczytywaliśmy wcześniej
sample_names = df.iloc[:, 0].values

# 2. Przygotowanie nazw kolumn dla wektorów embeddingów
# features.shape[1] to liczba wymiarów, którą wygenerował tGPT (np. 512 lub 768)
num_features = features.shape[1]
embedding_colnames = [f"emb{i+1}" for i in range(num_features)]

# 3. Stworzenie nowego DataFrame z embeddingami
# Wykorzystujemy zmienną 'features' (np.array), która powstała w Twoim notebooku
df_output = pd.DataFrame(features, columns=embedding_colnames)

# 4. Wstawienie kolumny z nazwami sampli na sam początek
# Ustawiamy nazwę kolumny na pusty ciąg znaków "", aby nagłówek był pusty
df_output.insert(0, "", sample_names)

# 5. Zapis do pliku CSV
# index=False zapobiega dopisywaniu dodatkowej kolumny z numeracją wierszy pandas
output_csv_path = "tgpt_embeddings_results_ov.csv"
df_output.to_csv(output_csv_path, index=False)

print(f"Sukces! Embeddingi zostały zapisane do: {output_csv_path}")
print(f"Kształt tabeli: {df_output.shape}")

Sukces! Embeddingi zostały zapisane do: tgpt_embeddings_results_ov.csv
Kształt tabeli: (422, 1025)
